# Lab 09 — Building a Simple RAG System

**DSA 8401 — Applied Machine Learning**

### Learning goals
By the end of this lab you should be able to:
1. Explain what **RAG** is and why we need it.
2. Split documents into **chunks**.
3. Turn text into **embeddings** (numbers that capture meaning).
4. **Search** for the chunks that best match a question.
5. Ask an **LLM** to answer using only those chunks.

**Core question:** can we make an AI answer questions about *our own* course notes — notes it has never seen before?

## What is RAG, and why do we need it?

An AI like ChatGPT only knows what it saw during training. It has **never seen your course
notes**. If you ask it about them, it will usually make something up that sounds confident
but is wrong. We call that a **hallucination**.

**RAG** stands for **R**etrieval-**A**ugmented **G**eneration. The idea is simple:

> Before asking the AI a question, first **find** the relevant notes and **paste them into
> the question**. Then the AI can read the answer instead of guessing.

It is like the difference between a closed-book exam and an open-book exam.

### The 5 steps we will build

```
   1. LOAD     read our notes from files
   2. CHUNK    cut them into small pieces
   3. EMBED    turn each piece into a list of numbers
   ------------------------------------------------ (done once)
   4. SEARCH   find the pieces that match the question
   5. ASK      give those pieces + the question to the AI
```

Steps 1–3 happen once. Steps 4–5 happen every time someone asks a question.

## 0. Setup

**Before you run anything:**

1. Get a free API key from **https://openrouter.ai/keys**
2. Copy the file `.env.example` to `.env` and paste your key inside it.
3. Choose the kernel **Python (AML Lab09 RAG)** at the top right of this notebook.

Never share your API key or upload it to GitHub.

In [1]:
# Step 1: Import the libraries we need
import os
import json
import numpy as np
import requests
from dotenv import load_dotenv

# Read the API key from the .env file
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")

if not API_KEY:
    print("No API key found! Copy .env.example to .env and put your key in it.")
else:
    print("API key loaded successfully.")

API key loaded successfully.


In [2]:
# Step 2: Settings -- all the choices we make are in this one cell

URL = "https://openrouter.ai/api/v1"

EMBED_MODEL = "perplexity/pplx-embed-v1-0.6b"   # turns text into numbers
CHAT_MODEL  = "google/gemini-3.8-flash"         # writes the answer

CHUNK_SIZE = 150    # how many words go in each chunk
TOP_K      = 3      # how many chunks we give to the AI
MAX_TOKENS = 1000    # longest answer we allow (also keeps the cost per call small)

# Every request to OpenRouter needs these headers
HEADERS = {
    "Authorization": "Bearer " + API_KEY,
    "Content-Type": "application/json",
}

print("Embedding model:", EMBED_MODEL)
print("Chat model:     ", CHAT_MODEL)
print("Max tokens:     ", MAX_TOKENS)

Embedding model: perplexity/pplx-embed-v1-0.6b
Chat model:      google/gemini-3.8-flash
Max tokens:      1000


### Quick test: is everything connected?

Let's send one tiny message to the AI before we build anything else. If this works, our key
and our internet connection are fine.

**If you see a `402` error**, it means your OpenRouter balance is low. OpenRouter checks
whether you could afford the *longest possible* reply before it runs anything, so we set
`MAX_TOKENS` in Step 2 to keep that ceiling small. Lower it further if the error persists,
or top up your credits.

In [4]:
# Step 3: Test that we can reach the AI
response = requests.post(
    URL + "/chat/completions",
    headers=HEADERS,
    json={
        "model": CHAT_MODEL,
        "messages": [{"role": "user", "content": "Say hello in 3 words"}],
        "max_tokens": 20,
    },
)

if response.status_code == 200:
    print("It works! The AI said:", response.json()["choices"][0]["message"]["content"])
else:
    print("Something went wrong:", response.status_code, response.text)

It works! The AI said: Hello


## 1. LOAD — read our notes

Our notes are in the `corpus/` folder. There are 10 small files: the course outline, a
summary of Labs 02 to 09, and a glossary.

These notes were written for this lab, so the AI has definitely never seen them. That makes
it easy to check whether RAG is really working.

In [5]:
# Step 4: Read every .md file in the corpus folder
import glob

documents = []

for filename in sorted(glob.glob("corpus/*.md")):
    with open(filename, encoding="utf-8") as f:
        text = f.read()
    name = os.path.basename(filename)
    documents.append({"name": name, "text": text})

print("Loaded", len(documents), "documents:")
for doc in documents:
    print("  ", doc["name"], "-", len(doc["text"].split()), "words")

Loaded 10 documents:
   01_course_outline.md - 242 words
   02_lab02_messy_data.md - 330 words
   03_lab03_evaluation.md - 313 words
   04_lab04_trees_optuna.md - 300 words
   05_lab05_unsupervised.md - 357 words
   06_lab06_mlp.md - 334 words
   07_lab07_cnn_xray.md - 358 words
   08_lab08_rnn.md - 360 words
   09_lab09_rag.md - 455 words
   10_glossary.md - 442 words


In [6]:
# Step 5: Look at one document so we know what we are working with
print(documents[7]["name"])
print("-" * 60)
print(documents[7]["text"][:700], "...")

08_lab08_rnn.md
------------------------------------------------------------
# Lab 08 — Recurrent Neural Networks: Forecasting COVID-19 Cases

Lab 08 forecasts daily new COVID-19 cases in the US from `Data/covid_19_data.csv`.

## Preparing the series

The raw file has one row per country/region per day with a **cumulative** confirmed-case
count. The preparation is: keep US rows and sum them (some days have several state rows),
take the day-to-day difference to convert cumulative counts into new cases per day, clip
negative values to zero (counts are sometimes corrected downward), and smooth with a 7-day
rolling average because reporting is uneven across the week — fewer cases are reported at
weekends.

## Windowing

A time series is turned into supervised `(X, y)` pa ...


## 2. CHUNK — cut the notes into small pieces

We do not search whole documents. We search small pieces called **chunks**.

**Why?** Imagine asking "What is an LSTM?" If we gave the AI a whole 400-word document, most
of it would be about other things. A small chunk gives the AI just the part it needs.

But chunks must not be *too* small either, or the answer gets cut in half.

| Chunk too small | Chunk too big |
|---|---|
| The answer gets split up and we lose half of it | Lots of unrelated text confuses the AI |

We will cut every document into pieces of **150 words**.

In [7]:
# Step 6: Cut a piece of text into chunks of CHUNK_SIZE words
def make_chunks(text, size=CHUNK_SIZE):
    words = text.split()
    chunks = []
    for start in range(0, len(words), size):
        piece = words[start:start + size]
        chunks.append(" ".join(piece))
    return chunks


# Try it on a short example first
example = "one two three four five six seven eight nine ten"
print(make_chunks(example, size=4))

['one two three four', 'five six seven eight', 'nine ten']


In [8]:
# Step 7: Chunk every document, remembering which file each chunk came from
chunks = []

for doc in documents:
    for piece in make_chunks(doc["text"]):
        chunks.append({"source": doc["name"], "text": piece})

print(len(documents), "documents became", len(chunks), "chunks")
print()
print("Here is chunk number 5:")
print("  from:", chunks[5]["source"])
print("  text:", chunks[5]["text"][:300], "...")

10 documents became 29 chunks

Here is chunk number 5:
  from: 03_lab03_evaluation.md
  text: # Lab 03 — Model Evaluation and Imbalanced Classes Lab 03 uses the cleaned loan feature table in `Data/CleanedFeaturesFromLoan.csv`. The lab is about choosing the right metric rather than about squeezing out a better model. ## Why accuracy fails On the loan default data roughly 6% of borrowers defau ...


## 3. EMBED — turn text into numbers

A computer cannot compare meanings, but it can compare numbers. An **embedding model** turns
a piece of text into a long list of numbers (a **vector**).

The clever part: texts that **mean** similar things get **similar numbers** — even if they
use completely different words.

For example, these two sentences share almost no words, but get very similar vectors:
- "How do I stop my model from overfitting?"
- "What prevents a network from memorising the training data?"

This is why RAG can find the right chunk even when the student's wording is different from
the notes.

In [11]:
# Step 8: Ask OpenRouter to turn a list of texts into vectors

def get_embeddings(texts):
    response = requests.post(
        URL + "/embeddings",
        headers=HEADERS,
        json={"model": EMBED_MODEL, "input": texts},
    )

    if response.status_code != 200:
        print("Error:", response.status_code, response.text)
        return None

    results = response.json()["data"]
    vectors = [item["embedding"] for item in results]
    return np.array(vectors)


# Try it with two sentences
test = get_embeddings(["I like cats", "I like dogs"])
print("Shape:", test.shape, " <- 2 sentences, each turned into", test.shape[1], "numbers")
print("First 8 numbers of sentence 1:", test[0][:8].round(3))

Shape: (2, 1024)  <- 2 sentences, each turned into 1024 numbers
First 8 numbers of sentence 1: [-0.148 -0.656  0.    -0.344 -0.047 -0.078 -0.352  0.398]


In [12]:
# Step 9: Turn ALL our chunks into vectors
# This takes a few seconds because we are sending them over the internet.

chunk_texts = [c["text"] for c in chunks]
chunk_vectors = get_embeddings(chunk_texts)

print("We now have a table of numbers:", chunk_vectors.shape)
print("That is", chunk_vectors.shape[0], "chunks, each described by",
      chunk_vectors.shape[1], "numbers")

We now have a table of numbers: (29, 1024)
That is 29 chunks, each described by 1024 numbers


## 4. SEARCH  find the chunks that match the question

Now we need to measure how similar two vectors are. We use **cosine similarity**, which
gives a score between -1 and 1:

- **close to 1** = very similar meaning
- **close to 0** = unrelated

The formula is easier than it looks. If we first make every vector have a length of 1
(this is called *normalising*), then cosine similarity is just multiplying the two vectors
together and adding up the result — a **dot product**.

In [16]:
# Step 10: Make every vector have length 1, so comparing them is easy

def normalise(vectors):
    lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / lengths


chunk_vectors = normalise(chunk_vectors)

# Check: every row should now have a length of 1.0
print("Lengths of the first 5 rows:", np.linalg.norm(chunk_vectors, axis=1)[:5].round(3))

Lengths of the first 5 rows: [1. 1. 1. 1. 1.]


In [17]:
# Step 11: Search -- turn the question into a vector, then score every chunk

def search(question, k=TOP_K):
    # 1. Turn the question into a vector (using the SAME model as the chunks!)
    question_vector = normalise(get_embeddings([question]))[0]

    # 2. Score every chunk at once
    scores = chunk_vectors @ question_vector

    # 3. Pick the k best ones
    best = np.argsort(scores)[::-1][:k]

    results = []
    for i in best:
        results.append({
            "source": chunks[i]["source"],
            "text":   chunks[i]["text"],
            "score":  round(float(scores[i]), 3),
        })
    return results

In [18]:
# Step 12: Try the search

question = "Why is an LSTM better than a SimpleRNN?"
found   = search(question)

print("QUESTION:", question)
print()
for i, hit in enumerate(found, 1):
    print(i, "| score:", hit["score"], "| from:", hit["source"])
    print("   ", hit["text"][:250], "...")
    print()

QUESTION: Why is an LSTM better than a SimpleRNN?

1 | score: 0.719 | from: 08_lab08_rnn.md
    held out as the test set. The split is **by date, never randomly** — a random split would let the model train on future values and predict the past, which is leakage. Scaling uses `MinMaxScaler` fitted on the training portion only. ## SimpleRNN versu ...

2 | score: 0.315 | from: 06_lab06_mlp.md
    # Lab 06 — Neural Networks: the Multi-Layer Perceptron Lab 06 is the first deep learning lab. It builds an MLP in Keras on the tabular loan data and compares it against the gradient-boosted baseline from Lab 04. ## Architecture The network is `Dense( ...

3 | score: 0.307 | from: 06_lab06_mlp.md
    tabular data, gradient boosting is usually still the right default.** Neural networks earn their keep on unstructured data — images, audio, text and sequences — which is what Labs 07, 08 and 09 cover. ...



Notice the scores. The top chunk should come from `08_lab08_rnn.md`, which is exactly where
we wrote about LSTMs. **The search found the right file without us telling it anything.**

Also notice we never mentioned the word "gates" or "vanishing gradient" in our question, yet
it still found the chunk that talks about them. That is the embedding doing its job.

## 5. ASK  let the AI write the answer

The final step. We build a message for the AI containing:

1. **Rules** — "only use the notes I give you, and say so if the answer isn't there"
2. **The chunks** we found
3. **The question**

Rule number 1 is the most important line in this whole notebook. It is what stops the AI
from making things up.

In [19]:
# Step 13: The rules we give the AI
RULES = """You are a teaching assistant for a machine learning course.

Answer the student's question using ONLY the notes provided below.

- Do not use any outside knowledge.
- Say which file each fact came from, like [08_lab08_rnn.md].
- If the notes do not contain the answer, reply exactly:
  "The course notes don't cover this."
- Keep the answer short - 3 or 4 sentences.
"""
print(RULES)

You are a teaching assistant for a machine learning course.

Answer the student's question using ONLY the notes provided below.

- Do not use any outside knowledge.
- Say which file each fact came from, like [08_lab08_rnn.md].
- If the notes do not contain the answer, reply exactly:
  "The course notes don't cover this."
- Keep the answer short - 3 or 4 sentences.



In [20]:
# Step 14: Send the notes + question to the AI
def ask_ai(question, notes):
    message = "NOTES:\n" + notes + "\n\nQUESTION: " + question

    response = requests.post(
        URL + "/chat/completions",
        headers=HEADERS,
        json={
            "model": CHAT_MODEL,
            "messages": [
                {"role": "system", "content": RULES},
                {"role": "user",   "content": message},
            ],
            "temperature": 0,     # 0 means "do not be creative, just use the notes"
            "max_tokens": MAX_TOKENS,
        },
    )

    if response.status_code != 200:
        return "Error: " + response.text

    return response.json()["choices"][0]["message"]["content"]

In [21]:
# Step 15: Put SEARCH and ASK together -- this is our finished RAG system!

def rag(question, k=TOP_K):
    
    
    # Step A: find the best chunks
    found = search(question, k)

    # Step B: join them into one block of text
    
    notes = ""
    for hit in found:
        notes += "[" + hit["source"] + "]\n" + hit["text"] + "\n\n"

    # Step C: let the AI answer
    answer = ask_ai(question, notes)

    # Show everything
    print("QUESTION:", question)
    print()
    print("ANSWER:")
    print(answer)
    print()
    print("(based on:", ", ".join(h["source"] for h in found), ")")

    return answer


_ = rag("Why is an LSTM better than a SimpleRNN?")

QUESTION: Why is an LSTM better than a SimpleRNN?

ANSWER:
A SimpleRNN suffers from the vanishing gradient problem because the gradient signal shrinks multiplicatively during backpropagation through time, preventing it from learning dependencies more than a few steps back [08_lab08_rnn.md]. In contrast, an LSTM introduces a cell state and three gates—forget, input, and output—to control what information is discarded, written, and read out [08_lab08_rnn.md]. This cell state provides an additive path for the gradient through time instead of a multiplicative one, allowing the LSTM to successfully learn long-range dependencies that a SimpleRNN cannot [08_lab08_rnn.md].

(based on: 08_lab08_rnn.md, 06_lab06_mlp.md, 06_lab06_mlp.md )


## 6. Let's try some questions

Our RAG system is finished. Now let's see how it behaves.

In [22]:
# Step 16: A question about Lab 02
_ = rag("Which columns in the fraud data cause data leakage?")

QUESTION: Which columns in the fraud data cause data leakage?

ANSWER:
The two columns that cause data leakage are `manual_review_score` and `settlement_status` [02_lab02_messy_data.md]. These are post-outcome fields that are only recorded after a transaction has been scored and investigated, meaning they are unavailable at prediction time [02_lab02_messy_data.md]. Including these leaky columns artificially inflates the model's ROC-AUC from approximately 0.63 to approximately 1.00 [02_lab02_messy_data.md].

(based on: 02_lab02_messy_data.md, 02_lab02_messy_data.md, 05_lab05_unsupervised.md )


In [22]:
# Step 17: A question about Lab 07
_ = rag("Why do we avoid flipping X-ray images left to right?")

QUESTION: Why do we avoid flipping X-ray images left to right?

ANSWER:
Horizontal flips are deliberately avoided because a chest X-ray has a definite left and right side [07_lab07_cnn_xray.md]. Since the heart sits on one side of the body, flipping the image left to right creates anatomically impossible training images [07_lab07_cnn_xray.md]. Therefore, this specific augmentation is excluded to preserve proper anatomical realism during training [07_lab07_cnn_xray.md].

(based on: 07_lab07_cnn_xray.md, 07_lab07_cnn_xray.md, 07_lab07_cnn_xray.md )


### The most important test: a question our notes cannot answer

The search **always** returns 3 chunks. It has no way of saying "I found nothing useful" —
it just returns the 3 least-bad matches.

So what happens if we ask something that is not in our notes at all? Without our RULES, the
AI would happily invent an answer. Let's check that it refuses instead.

In [23]:
# Step 18: Ask about something that is NOT in our notes
_ = rag("What is the best recipe for chicken biryani?")

QUESTION: What is the best recipe for chicken biryani?

ANSWER:
The course notes don't cover this.

(based on: 04_lab04_trees_optuna.md, 06_lab06_mlp.md, 06_lab06_mlp.md )


It said it doesn't know. **That refusal is a feature, not a failure.** A system that admits
it doesn't know is far more useful than one that confidently makes something up.

Without the notes, the AI either guesses a number or admits it doesn't know. With the notes,
it gives the correct answer (**0.63**) and tells you which file it came from — so you can
open `corpus/02_lab02_messy_data.md` and check it yourself.

**That is the whole point of RAG.** The AI did not get smarter. It is the exact same model.
We just gave it the right page to read.

##  Your turn

Change the question below and run the cell. Try to **break it**:

- Ask something vague
- Ask about two labs in one question
- Ask something that is *almost* in the notes but not quite

Then look at which chunks came back, and think about *why* it behaved that way.

In [25]:
# Step 22: Change this question and re-run!

my_question = "What is overfitting?"

_ = rag(my_question)

QUESTION: What is overfitting?

ANSWER:
Overfitting occurs when a model has memorised the training data, including its noise [10_glossary.md]. It is diagnosed by observing a large gap between training and validation performance [10_glossary.md]. For example, gradient boosting models will overfit if the learning rate is too high or if there are too many estimators [04_lab04_trees_optuna.md].

(based on: 10_glossary.md, 04_lab04_trees_optuna.md, 04_lab04_trees_optuna.md )


## 10. Exercises

Write your answers in new cells below. Include the code **and** a short explanation in your
own words.

**1. Change the chunk size.**
Set `CHUNK_SIZE = 50` in Step 2, then re-run Steps 6, 7, 9 and 10 (you must re-chunk AND
re-embed). Run the hit-rate test in Step 20 again. Did it get better or worse? Now try
`CHUNK_SIZE = 400`. Write one paragraph explaining what you found.

**2. Add your own document.**
Create a new file in the `corpus/` folder about any topic you like. Re-run the notebook from
Step 4. Ask a question that only your new file can answer. Does it find it?

**3. Add three test questions.**
Add 3 more questions to `test_questions` in Step 20 and re-run it. Try to write one question
you think the search will get **wrong**. Were you right?

**4. Break the rules.**
In Step 13, delete the line that tells the AI to say "The course notes don't cover this".
Re-run Step 18 (the biryani question). What happens now? Put the line back afterwards, and
explain in one sentence why that line matters.

---